Ce notebook ne contient des tests que sur une seule entrée du Wiktionnaire : hippique. J'ai testé plusieurs modèles sur cette seule entrée pour trouver le meilleur en termes de performance/temps/mémoire.

# Installation et configuration

In [ ]:
# Uniquement pour Colab, sinon il faut juste télécharger Ollama via ce lien : https://ollama.com/download
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,813 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-secu

In [ ]:
# Uniquement pour Colab
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!pip install ollama

In [ ]:
import ollama
import json

# Tests

In [ ]:
mydict = {
    "hippique" : {"catégorie" : "Adjectif", "définition" : "(Équitation) Relatif au cheval, à l’équitation ou à l’hippologie.", "étymologie" : "(1842) Du grec ancien ἱππικός, hippikós (même sens), de ίππος, hippos (« cheval »)."}
    }

with open('example.json', 'w', encoding='utf-8') as f:
    json.dump(mydict, f, ensure_ascii=False, indent=2)

with open('example.json', 'r', encoding='utf-8') as f:
    json_entry = f.read()

SYSTEM_PROMPT = """Tu es un assistant qui répond efficacement aux questions
à partir d'un fichier JSON qui contient plusieurs informations à propos d'un mot.
Tu ne dois pas donner d'explications, tu ne dois pas raisonner, tu ne dois pas reformuler la tâche."""

prompt = f"""l'enregistrement en JSON suivant : {json_entry} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
    * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
    * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
    * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou NULL si l'information est absente)
    * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou NULL si l'information est absente)
    * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (liste de composants ou NULL si l'information est absente)
    * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou NULL si l'information est absente)"""



## Sur Qwen

In [ ]:
# Test avec le modèle Qwen
!ollama pull qwen3:0.6b

In [ ]:
import pandas as pd

In [ ]:
response = ollama.chat(
        model="qwen3:0.6b",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
        format = "json"
    )
print(f"Réponse : {response['message']['content']}")

Réponse : {"hippique": {"catégorie": "Adjectif", "définition": "(Équitation) Relatif au cheval, à l’équitation ou à l’hiphologie.", "étymologie": "Du grec ancien ἱππικός, hippikós (même sens), de ίππος, hiphos (« cheval »)."}}


In [ ]:
content = response['message']['content']
data = json.loads(content)
df = pd.DataFrame([data])
df

,hippique
0,"{'catégorie': 'Adjectif', 'définition': '(Équi..."


|Qwen | Juste | Faux |
|-|------|---|
| Q1 |   | x |
| Q2 | x |   |
| Q3 |   | x |
| Q4 |   | x |
| Q5 |   | x |
| Q6 |   | x |
Note : 1/6

Remarque : il répond en anglais "French".

## Sur GPT-oss

In [ ]:
# Test avec le modèle GPT-oss
!ollama pull gpt-oss

In [ ]:
response = ollama.chat(
        model="gpt-oss",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
    )
print(f"Réponse : {response['message']['content']}")

Réponse : ```json
{
  "Q1": "grec",
  "Q2": "hippos",
  "Q3": null,
  "Q4": "ikos",
  "Q5": ["hippos"],
  "Q6": "suffixation"
}
```



| GPT-oss | Juste | Faux |
|-|------|---|
| Q1 | x |   |
| Q2 | x |   |
| Q3 | x |   |
| Q4 |   | x |
| Q5 |   | x |
| Q6 | x |   |
Note : 4/6

## Sur Phi-3

In [ ]:
!ollama pull phi3

In [ ]:
response = ollama.chat(
        model="phi3",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
    )
print(f"Réponse : {response['message']['content']}")

Réponse : {
  "enquête_sur_le_mot": {
    "mot_en_question": "hippique",
    "langue_origine": "grec",
    "base": "hippos",
    "préfixe": null,
    "suffixe": null,
    "composants": null,
    "morphologie": null
  }
}



Tu es un assistant qui répond efficacement aux questions
à partir d'un fichier JSON qui contient plusieurs informations à propos d'un mot.
Tu ne dois pas donner d'explications, tu ne dois pas raisonner, tu ne dois pas reformuler la tâche.


| Phi-3 | Juste | Faux |
|-|------|---|
| Q1 | x |   |
| Q2 |   | x (hippos plutôt ?) |
| Q3 | x |   |
| Q4 | x |   |
| Q5 |   | x |
| Q6 | x |   |
Note : 4.5/6

Remarque 1 : il a inventé les clés du JSON "mot", "lng_origine", "préfixe"... Et je n'ai jamais demandé le mot d'ailleurs.

Remarque 2 : il a presque bien relevé la base.

## Sur Gemma-3 4B

In [ ]:
!ollama pull gemma3:4b

In [ ]:
response = ollama.chat(
        model="gemma3:4b",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
    )
print(f"Réponse : {response['message']['content']}")

Réponse : ```json
{
  "hippique": {
    "Q1": "grec",
    "Q2": "hippos",
    "Q3": "NULL",
    "Q4": "NULL",
    "Q5": ["ἱππικός"],
    "Q6": "suffixation"
  }
}
```


| Gemma-3 4B | Juste | Faux |
|-|------|---|
| Q1 | x |   |
| Q2 | x |   |
| Q3 | x |   |
| Q4 |   | x |
| Q5 |   | x |
| Q6 | x |   |
Note : 4/6

## Sur Orca-mini

In [ ]:
!ollama pull orca-mini

In [ ]:
response = ollama.chat(
        model="orca-mini",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
    )
print(f"Réponse : {response['message']['content']}")

Réponse :  Q1: The language of origin of the word mentioned in the etymology is French. 
Q2: The base of the word mentioned in the etymology is "equus" (Latin for "horse"). 
Q3: No prefixe is mentioned in the etymology. 
Q4: No suffix is mentioned in the etymology. 
Q5: The following components are mentioned in the etymology: "hippique" ("hippology"). 
Q6: The type of procédé morphologique mentioned in the etymology is suffixation.


| Orca-mini | Juste | Faux |
|-|------|---|
| Q1 |   | x |
| Q2 |   | x |
| Q3 |   | x |
| Q4 |   | x |
| Q5 |   | x |
| Q6 |   | x |
Note : 0/6

Remarque 1 : n'a pas créé un fichier JSON comme demandé.

Remarque 2 : a répété les questions presque mot pour mot pour y répondre => "The base of the word mentioned in the eurymology is "Latin" **or "NULL" if the information is absent**." ce qui donne des réponses très chelou. Ne sait même pas écrire étymologie (il a inventé un autre mot "eurymology"). Parfois il a juste reformulé la question => "If a prefix is mentioned in the etymology, what is this prefix? (One word or NULL if the information is absent.)". Et quand il y a un élément de réponse, cet élément est faux (le latin n'est pas la langue d'origine, ni la base).

Remarque 3 : répond en anglais.

## Sur Gemma 7B

In [ ]:
!ollama pull gemma:7b

In [ ]:
response = ollama.chat(
        model="gemma:7b",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
    )
print(f"Réponse : {response['message']['content']}")

Réponse : ```json
{
  "hippique": {
    "catégorie": "Adjectif",
    "définition": "(Équitation) Relatif au cheval, à l’équitation ou à l’hippologie.",
    "étymologie": "(1842) Du grec ancien ἱππικός, hippikós (même sens), de ίππος, hippos («cheval »)."
  },
  "questions": [
    { "question": "Quelle est la langue d'origine du mot ?", "réponse": "grec ancien" },
    { "question": "Quelle est la base du mot ?", "réponse": "hippos" },
    { "question": "Y a-t-il un préfixe ?", "réponse": NULL },
    { "question": "Y a-t-il un suffixe ?", "réponse": NULL },
    { "question": "Y a-t-il des composants ?", "réponse": NULL },
    { "question": "Quel est le type du procédé morphologique ?", "réponse": "Composition" }
  ]
}
```


| Gemma 7B | Juste | Faux |
|-|------|---|
| Q1 | x |   |
| Q2 | x |   |
| Q3 | x |   |
| Q4 |   | x |
| Q5 | x |   |
| Q6 |   | x |
Note : 4/6

Remarque : a ajouté les questions à mon fichier JSON de base alors que je voulais qu'il me créé un nouvel enregistrement de JSON sans le mot hippique etc.

(avec du recul, mon prompt n'était pas assez précis pour qu'il comprenne les choses comme je les voyais)

## Sur DeepSeek-R1 7B

In [ ]:
!ollama pull deepseek-r1:7b

In [ ]:
response = ollama.chat(
        model="deepseek-r1:7b",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {"role": "user", "content": prompt},
        ],
    )
print(f"Réponse : {response['message']['content']}")

| DeepSeek-R1 7B | Juste | Faux |
|-|------|---|
| Q1 |   | x |
| Q2 | x |   |
| Q3 | x |   |
| Q4 |   | x |
| Q5 | x |   |
| Q6 | x |   |
Note : 3.5/6

Remarque 1 : invente des mots "Hieracote", "prifixe".

Remarque 2 : a fait un peu comme Gemma 7B, s'est inspiré de mon fichier JSON de base mais l'a réécrit.

## Conclusions

| Modèle     | Note |
|------------|------|      
| Phi-3      | ⭐⭐⭐⭐⭐ |
| GPT-oss    | ⭐⭐⭐⭐   |
| Gemma-3 4B | ⭐⭐⭐⭐   |
| Gemma 7B   | ⭐⭐⭐⭐   |
| DeepSeek   | ⭐⭐        |
| Qwen       | ⭐           |
| Orca-mini  |              |

Notes : le prompt a été changé ==> juste quelques reformulations concernant l'introduction du fichier json_entry dans le prompt et la façon de dire que je veux au format JSON en output. J'ai mis le fait qu'il ne devait pas raisonner, ni reformuler etc. dans le system prompt et non pas dans le prompt. (j'ai pensé que c'était là sa place puisque je me suis inspirée d'un TD de cours et que dans ce TD le prof avait fait comme ça, c'est peut-être quelque chose à changer : à voir). J'ai également ajouté un champ "définition" dans le json_entry.